In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# An EBNF based Parser for Arithmetic Expressions

In this notebook we implement an <span style="font-variant:small-caps;">Ebnf</span> recursive-descend parser for arithmetic expressions.  This parser implements the following <span style="font-variant:small-caps;">Ebnf</span> grammar:
$$
  \begin{eqnarray*}
  \mathrm{expr}    & \rightarrow & \mathrm{product}\;\;\bigl((\texttt{'+'}\;|\;\texttt{'-'})\;\; \mathrm{product}\bigr)^* \\[0.2cm]
  \mathrm{product} & \rightarrow & \mathrm{factor} \;\;\bigl((\texttt{'*'}\;|\;\texttt{'/'})\;\; \mathrm{factor}\bigr)^*  \\[0.2cm]   
  \mathrm{factor}  & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}                             \\
                   & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

## The Scanner

In [ ]:
open System
open System.Globalization
open System.Text.RegularExpressions

// \G anchors every match at the previous match: illegal characters cannot disappear.
let tokenize (text: string) =
    let pattern = Regex(@"\G(?:\s+|[1-9][0-9]*|0|[a-z][0-9]*|[-+*/()])")
    let rec scan position tokens =
        if position = text.Length then List.rev tokens
        else
            let token = pattern.Match(text, position)
            if not token.Success then failwithf "Illegal character at column %d" (position + 1)
            let rest = position + token.Length
            if String.IsNullOrWhiteSpace token.Value then scan rest tokens
            else scan rest (token.Value :: tokens)
    scan 0 []

The scanner returns strings and rejects unrecognized characters. Variables consist of a lower-case letter followed by optional digits.

In [ ]:
printfn "%A" (tokenize "12 * x + y1 * 4 / 6 - z3")

## Implementing the Recursive Descend Parser

The discriminated union `AST` makes every supported node explicit.

The mutually recursive parser group uses `let rec` and `and`; no forward declarations are needed.

The function `parse` takes a string `s` as input and parses this string according to the recursive grammar
shown above.

The function `parseExpr(TL)` takes a list of tokens `TL` and tries to parse an expresssion according to the following
<span style="font-variant:small-caps;">Ebnf</span> grammar rule:
$$ \mathrm{expr} \;\rightarrow\; \mathrm{product}\;\;\bigl((\texttt{'+'}\;|\;\texttt{'-'})\;\; \mathrm{product}\bigr)^* $$
It returns an *abstract syntax tree* representing the expression and a list of all the tokens that have not been consumed during parsing.

The function `parseProduct(TL)` takes a list of tokens `TL` and tries to parse a product according to the following
<span style="font-variant:small-caps;">Ebnf</span> grammar rule:
$$ \mathrm{product} \;\rightarrow\; \mathrm{factor} \;\;\bigl((\texttt{'*'}\;|\;\texttt{'/'})\;\; \mathrm{factor}\bigr)^*  $$
It returns the value of the product and a list of all the tokens that have not been consumed during parsing.

The function `parseFactor` implements the following grammar rules:
$$
  \begin{eqnarray*}
  \mathrm{factor}      & \;\rightarrow\; & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}    \\
                       & \;\mid          & \;\texttt{NUMBER}                                  \\
                       & \;\mid          & \;\texttt{VARIABLE}
  \end{eqnarray*}
$$

It takes one argument:
- `TL` is the list of tokens that still need to be consumed.

It returns a pair of the form `(value, Rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `TL` and
- `Rest` is a list of those tokens that have not been consumed while trying to parse a factor.

In [ ]:
type BinaryOp = Add | Subtract | Multiply | Divide
type AST = Number of int | Variable of string | Binary of BinaryOp * AST * AST

let rec parseExpr tokens =
    let first, rest = parseProduct tokens
    let rec repeat tree remaining =
        match remaining with
        | ("+" | "-" as op) :: tail ->
            let right, rest = parseProduct tail
            repeat (Binary((if op = "+" then Add else Subtract), tree, right)) rest
        | _ -> tree, remaining
    repeat first rest
and parseProduct tokens =
    let first, rest = parseFactor tokens
    let rec repeat tree remaining =
        match remaining with
        | ("*" | "/" as op) :: tail ->
            let right, rest = parseFactor tail
            repeat (Binary((if op = "*" then Multiply else Divide), tree, right)) rest
        | _ -> tree, remaining
    repeat first rest
and parseFactor tokens =
    match tokens with
    | "(" :: tail ->
        let tree, rest = parseExpr tail
        match rest with
        | ")" :: remaining -> tree, remaining
        | _ -> failwith "Parse error: expected ')'"
    | token :: tail when Regex.IsMatch(token, @"^[0-9]+$") -> Number(int token), tail
    | token :: tail when Regex.IsMatch(token, @"^[a-z][0-9]*$") -> Variable token, tail
    | _ -> failwith "Parse error: expected a factor"

let parse text =
    let tree, rest = parseExpr (tokenize text)
    if not rest.IsEmpty then failwithf "Unconsumed tokens: %A" rest
    tree

let rec evaluate values tree =
    match tree with
    | Number n -> float n
    | Variable name -> Map.find name values
    | Binary(op, left, right) ->
        let l, r = evaluate values left, evaluate values right
        match op with
        | Add -> l + r | Subtract -> l - r | Multiply -> l * r | Divide -> divide l r

## Testing

The typed tree can be evaluated with a variable environment and converted into a Graphviz tree.

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

In [ ]:
let rec toTree = function
    | Number n -> Leaf(string n)
    | Variable v -> Leaf v
    | Binary(op, left, right) ->
        let label = match op with Add -> "+" | Subtract -> "-" | Multiply -> "*" | Divide -> "/"
        Node(label, [toTree left; toTree right])

In [ ]:
let tree = parse "12 * x + y1 * 4 / 6 - z3"
printfn "%A" tree
printfn "%s" (treeToDot (toTree tree))
checkNotebook (evaluate (Map.ofList ["x",2.0; "y1",3.0; "z3",1.0]) tree = 25.0)

In [ ]:
let rejects text =
    try parse text |> ignore; false
    with _ -> true
checkNotebook (rejects "1 + @ 2")
checkNotebook (rejects "1 +")
checkNotebook (rejects "(1 + 2")
checkNotebook (rejects "1 2")

### Rendered syntax tree

The diagram makes the nesting and operator structure visible in Jupyter.

In [ ]:
let svg = toTree tree |> treeToDot |> renderDot
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(svg), "text/html") |> ignore
